# Build Silver Delta Table (Pipeline-Invoked)

**Called by:** `PL_ProducerComp_Ingest` pipeline  
**Expects:** Pipeline has already completed:
1. `staging_producer_payments` — Delta table with regular columns (no blobs)
2. `Files/payment_details/{PaymentID}.json` — JSON blob files
3. `Files/attachments/{PaymentID}_attachment.bin` — Binary attachment files

**This notebook:** Reads the staging table, constructs file path pointers, and MERGE-upserts into the Silver Delta table.

In [ ]:
import os

# ── Config ──
lakehouse_base = "/lakehouse/default"
staging_table = "staging_producer_payments"
silver_table  = "producer_payments_silver"

payment_details_dir = os.path.join(lakehouse_base, "Files", "payment_details")
attachments_dir     = os.path.join(lakehouse_base, "Files", "attachments")

In [ ]:
from pyspark.sql.functions import col, lit, concat, when
from pyspark.sql import functions as F

# Read staging table (regular columns only — written by Copy Activity)
df_staging = spark.table(staging_table)
print(f"Staging rows: {df_staging.count()}")
df_staging.show(truncate=False)

In [ ]:
# ── Build file existence lookup from the Files section ──
# Scan what the pipeline's ForEach actually wrote

detail_files = set(os.listdir(payment_details_dir)) if os.path.exists(payment_details_dir) else set()
attach_files = set(os.listdir(attachments_dir)) if os.path.exists(attachments_dir) else set()

print(f"PaymentDetails files found: {len(detail_files)}")
print(f"Attachment files found:     {len(attach_files)}")

In [ ]:
# ── Construct Silver DataFrame: staging cols + file path pointers ──
# PaymentDetails pointer: Files/payment_details/{PaymentID}.json
# Attachment pointer:     Files/attachments/{PaymentID}_attachment.bin (if file exists)

# Build lookup of which PaymentIDs have attachment files
attach_pids = set()
for fname in attach_files:
    # File format: {PaymentID}_attachment.bin
    try:
        pid = int(fname.split("_")[0])
        attach_pids.add(pid)
    except ValueError:
        pass

# Add pointer columns to staging data
df_silver = df_staging.withColumn(
    "PaymentDetailsFilePath",
    concat(lit("Files/payment_details/"), col("PaymentID").cast("string"), lit(".json"))
).withColumn(
    "AttachmentFilePath",
    when(
        col("PaymentID").isin(list(attach_pids)),
        concat(lit("Files/attachments/"), col("PaymentID").cast("string"), lit("_attachment.bin"))
    ).otherwise(lit(None))
)

print("Silver DataFrame with file pointers:")
df_silver.show(truncate=False)

In [ ]:
from delta.tables import DeltaTable

# ── MERGE into Silver Delta table ──
if not spark.catalog.tableExists(silver_table):
    df_silver.write.format("delta").saveAsTable(silver_table)
    print(f"Created Silver Delta table: {silver_table}")
else:
    delta_target = DeltaTable.forName(spark, silver_table)
    delta_target.alias("t").merge(
        df_silver.alias("s"),
        "t.PaymentID = s.PaymentID"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print(f"Merged into Silver Delta table: {silver_table}")

# Final output
print(f"\n=== Silver Table: {silver_table} ===")
spark.sql(f"""
    SELECT PaymentID, ProducerID, CarrierName, PaymentDate, GrossAmount,
           PaymentDetailsFilePath, AttachmentFilePath
    FROM {silver_table} ORDER BY PaymentID
""").show(truncate=False)

In [ ]:
# ── Validation: verify file pointers resolve to real files ──
print("=== File Pointer Validation ===\n")
silver_rows = spark.table(silver_table).collect()
valid = 0
missing = 0

for row in silver_rows:
    pid = row["PaymentID"]
    
    # Check PaymentDetails file
    dp = row["PaymentDetailsFilePath"]
    dp_exists = os.path.exists(os.path.join(lakehouse_base, dp)) if dp else False
    
    # Check Attachment file
    ap = row["AttachmentFilePath"]
    ap_exists = os.path.exists(os.path.join(lakehouse_base, ap)) if ap else True  # NULL is OK
    
    status = "✓" if (dp_exists and ap_exists) else "✗"
    if dp_exists and ap_exists:
        valid += 1
    else:
        missing += 1
    
    print(f"  {status} PaymentID {pid}: details={dp_exists}, attachment={'N/A' if not ap else ap_exists}")

print(f"\nValid: {valid}, Missing: {missing}")